
### Ziel dieser Datei
Endprodukt für unsere Zielgruppe erstellen. Eine interaktive Webkarte, auf der man z.B. nach Kantonen filtern kann und wenn möglich auch nach zB. Abstand zu Infrastruktur etc. Dann sollen die potenziellen Lagerplätze evtl. inklusive nützlicher Zusatzinfos (Pop-ups) angezeigt werden.

### Datenquellen
* Die aus Datei 2 resultiere GeoJSON-Datei wo alle potenziellen Lagerplätze drin sind
* Evtl. GeoJSON-Dateien aus schritt 1 wo die Infrastruktur-Objekte mit Koordinaten zeigt (könnte man zB für die Karstendarstellung brauchen, um dann auf der Karte diese Objekte mit einem Punkt oder so hervorzuheben)

### Grober Codeaufbau
1. Koordinaten zurücktransformieren: Weil Folium benötigt Weltkoordinaten (WGS84). Also muss von LV95 wieder zurück in WGS84 transformiert werden. 
2. Kanton-Filter: Einen Filter einbauen (z.B. Pandas-Abfrage), damit Folium Karte nicht immer komplette Schweiz laden muss. so läuft es schneller.
3. Folium-Karte zusammenbauen: Folium Map laden, entsprechende Flächen aus GeoJSON hinzufügen. evtl. Markers für die Infrastrukturobjekte aus GeoJSON Dateien aus schitt 1 anzeigen. Evtl. Pop-Ups mit texten erstellen. 
4. Wenn möglich Puffer-Filter für die Abstände zu Infrastruktur zu definieren (damit man einstellen kann 100m, 500m, 1000m, 2000m, >2000m
)
### Export & Übernahme für die Nächste Datei 4
* Fertige Karte als eigenständige Webseite exportieren im html-format. Das sollte dann in Website geöffnet werden können (gut für Pitch)

### Import & Einrichtung

In [6]:
import json
import geopandas as gpd
import pandas as pd
import folium
from folium import plugins
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✅ Alle Libraries erfolgreich importiert")

✅ Alle Libraries erfolgreich importiert


## 1. Daten laden

In [7]:
# Testdatei laden
data_path = 'output/geeignete_lagerflaechen_BL.geojson'

print(f"Lade: {data_path}...")
gdf = gpd.read_file(data_path)

print(f"\n✅ Daten geladen!")
print(f"   Features: {len(gdf)}")
print(f"   CRS: {gdf.crs}")
print(f"\n📋 Spalten:")
for col in gdf.columns:
    print(f"   - {col}")

print(f"\n🎯 Erste Feature:")
print(gdf.iloc[0])

Lade: output/geeignete_lagerflaechen_BL.geojson...

✅ Daten geladen!
   Features: 1602
   CRS: EPSG:2056

📋 Spalten:
   - lagerplatz_id
   - landuse
   - flaeche_m2
   - flaeche_ha
   - dist_bauernhof_m
   - dist_hydrant_m
   - dist_oev_m
   - nahe_bauernhof
   - nahe_hydrant
   - nahe_oev
   - filter_score
   - bewertung
   - geometry

🎯 Erste Feature:
lagerplatz_id                                                       1
landuse                                                        forest
flaeche_m2                                             15519248.89722
flaeche_ha                                                 1551.92489
dist_bauernhof_m                                                  0.0
dist_hydrant_m                                                   None
dist_oev_m                                                        0.0
nahe_bauernhof                                                   True
nahe_hydrant                                                    False
nahe_oev      

In [8]:
print("\n📊 DATENEXPLORATION")
print("="*60)

# Attribut-Statistiken
print(f"\nFlächenstatistiken (flaeche_m2):")
print(f"  Minimum: {gdf['flaeche_m2'].min():,.0f} m²")
print(f"  Maximum: {gdf['flaeche_m2'].max():,.0f} m²")
print(f"  Durchschnitt: {gdf['flaeche_m2'].mean():,.0f} m²")
print(f"  Median: {gdf['flaeche_m2'].median():,.0f} m²")
print(f"  Gesamt: {gdf['flaeche_m2'].sum():,.0f} m² = {gdf['flaeche_ha'].sum():,.0f} ha")

# Landnutzung
print(f"\nVerteilung Landnutzung:")
print(gdf['landuse'].value_counts())

# Bewertung
print(f"\nVerteilung Bewertung:")
print(gdf['bewertung'].value_counts())

# Nähe zu Infrastruktur
print(f"\nNähe zu Infrastruktur:")
print(f"  Nahe Bauernhof: {gdf['nahe_bauernhof'].sum()} ({gdf['nahe_bauernhof'].sum()/len(gdf)*100:.1f}%)")
print(f"  Nahe ÖV: {gdf['nahe_oev'].sum()} ({gdf['nahe_oev'].sum()/len(gdf)*100:.1f}%)")

# Bounds
print(f"\n🗺️ Geografische Ausdehnung:")
bounds = gdf.total_bounds
print(f"  X: {bounds[0]:.0f} - {bounds[2]:.0f}")
print(f"  Y: {bounds[1]:.0f} - {bounds[3]:.0f}")


📊 DATENEXPLORATION

Flächenstatistiken (flaeche_m2):
  Minimum: 5,004 m²
  Maximum: 31,024,414 m²
  Durchschnitt: 273,131 m²
  Median: 27,588 m²
  Gesamt: 437,555,595 m² = 43,756 ha

Verteilung Landnutzung:
landuse
meadow    1063
forest     539
Name: count, dtype: int64

Verteilung Bewertung:
bewertung
geeignet                 1501
bedingt geeignet           98
nur Must-have erfüllt       3
Name: count, dtype: int64

Nähe zu Infrastruktur:
  Nahe Bauernhof: 1559 (97.3%)
  Nahe ÖV: 1541 (96.2%)

🗺️ Geografische Ausdehnung:
  X: 2585399 - 2641891
  Y: 1240985 - 1267697


## 2. Koordinaten-Transformation: LV95 → WGS84

In [9]:
print(f"Original CRS: {gdf.crs}")

# Transformation
gdf = gdf.to_crs('EPSG:4326')

print(f"Neu CRS: {gdf.crs}")
print("✅ Transformation LV95 → WGS84 abgeschlossen")

# Zentrum berechnen
bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

print(f"\n🎯 Kartenzentrum: Lat {center_lat:.4f}, Lon {center_lon:.4f}")

Original CRS: EPSG:2056
Neu CRS: EPSG:4326
✅ Transformation LV95 → WGS84 abgeschlossen

🎯 Kartenzentrum: Lat 47.4396, Lon 7.6195


## 3. Folium-Karte bauen

In [10]:
# Karte erstellen
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles='OpenStreetMap',
    prefer_canvas=True  # Performance-Optimization
)

print("✅ Basis-Karte erstellt")

✅ Basis-Karte erstellt


In [11]:
# Farben nach Landnutzung
FARBEN = {
    'forest': {'fill': '#2d5016', 'outline': '#1a3009', 'name': '🌲 Wald'},
    'meadow': {'fill': '#90EE90', 'outline': '#228B22', 'name': '🌾 Wiese'},
    'other': {'fill': '#FFD700', 'outline': '#FFA500', 'name': '⚪ Sonstige'}
}

# Filter-Score Farben
FILTER_FARBEN = {
    3: '#00AA00',  # Grün - sehr geeignet
    2: '#FFAA00',  # Orange - geeignet
    1: '#FF0000',  # Rot - bedingt geeignet
    0: '#888888'   # Grau - nicht geeignet
}

print("✅ Farb-Palette definiert")

✅ Farb-Palette definiert


In [12]:
def create_popup(row):
    """Erstelle HTML-Popup mit Informationen"""
    html = f"""
    <div style="width: 280px; font-family: Arial; font-size: 12px;">
        <h4 style="margin-top: 0; color: #2d5016;">🗺️ Lagerfläche #{row.get('lagerplatz_id', '?')}</h4>
        
        <table style="width: 100%; border-collapse: collapse;">
            <tr style="background: #f0f0f0;">
                <td style="padding: 5px; border: 1px solid #ccc;"><b>Fläche:</b></td>
                <td style="padding: 5px; border: 1px solid #ccc;">{row.get('flaeche_ha', 0):.2f} ha</td>
            </tr>
            <tr>
                <td style="padding: 5px; border: 1px solid #ccc;"><b>Typ:</b></td>
                <td style="padding: 5px; border: 1px solid #ccc;">{row.get('landuse', 'unbekannt')}</td>
            </tr>
            <tr style="background: #f0f0f0;">
                <td style="padding: 5px; border: 1px solid #ccc;"><b>Bewertung:</b></td>
                <td style="padding: 5px; border: 1px solid #ccc;"><b>{row.get('bewertung', '?')}</b></td>
            </tr>
            <tr>
                <td style="padding: 5px; border: 1px solid #ccc;"><b>Score:</b></td>
                <td style="padding: 5px; border: 1px solid #ccc;">{row.get('filter_score', '?')}/3</td>
            </tr>
        </table>
        
        <hr style="margin: 10px 0; border: 1px solid #ddd;">
        
        <div style="font-size: 11px; color: #666;">
            <p><b>Infrastruktur:</b></p>
            <p>🚜 Bauernhof: {row.get('dist_bauernhof_m', -1):.0f}m {'✅' if row.get('nahe_bauernhof', False) else '❌'}</p>
            <p>🚌 ÖV: {row.get('dist_oev_m', -1):.0f}m {'✅' if row.get('nahe_oev', False) else '❌'}</p>
        </div>
    </div>
    """
    return folium.Popup(html, max_width=300)

print("Popup-Funktion definiert. Füge Features hinzu...")
print(f"Verarbeite {len(gdf)} Features...\n")

# Feature-Group für Lagerplätze
fg_lagerflaechen = folium.FeatureGroup(name='Geeignete Lagerplätze', show=True)

# Fortschritts-Counter
for i, (idx, row) in enumerate(gdf.iterrows()):
    geom = row.geometry
    
    # Nur Polygone verarbeiten
    if geom.geom_type != 'Polygon':
        continue
    
    # Koordinaten extrahieren
    coords = list(geom.exterior.coords)
    locations = [[lat, lon] for lon, lat in coords]
    
    # Farbe bestimmen (nach Landnutzung)
    landuse = row.get('landuse', 'other')
    farbe_config = FARBEN.get(landuse, FARBEN['other'])
    
    # Polygon zeichnen
    folium.Polygon(
        locations=locations,
        color=farbe_config['outline'],
        fill=True,
        fillColor=farbe_config['fill'],
        fillOpacity=0.7,
        weight=2,
        popup=create_popup(row),
        tooltip=f"ID: {row.get('lagerplatz_id', idx)} - {landuse}"
    ).add_to(fg_lagerflaechen)
    
    # Fortschritt anzeigen
    if (i + 1) % 100 == 0:
        print(f"  ✓ {i + 1}/{len(gdf)} Features verarbeitet")

fg_lagerflaechen.add_to(m)
print(f"\n✅ Alle {len(gdf)} Features zur Karte hinzugefügt")

Popup-Funktion definiert. Füge Features hinzu...
Verarbeite 1602 Features...

  ✓ 100/1602 Features verarbeitet
  ✓ 200/1602 Features verarbeitet
  ✓ 300/1602 Features verarbeitet
  ✓ 400/1602 Features verarbeitet
  ✓ 500/1602 Features verarbeitet
  ✓ 600/1602 Features verarbeitet
  ✓ 700/1602 Features verarbeitet
  ✓ 800/1602 Features verarbeitet
  ✓ 900/1602 Features verarbeitet
  ✓ 1000/1602 Features verarbeitet
  ✓ 1100/1602 Features verarbeitet
  ✓ 1200/1602 Features verarbeitet
  ✓ 1300/1602 Features verarbeitet
  ✓ 1400/1602 Features verarbeitet
  ✓ 1500/1602 Features verarbeitet
  ✓ 1600/1602 Features verarbeitet

✅ Alle 1602 Features zur Karte hinzugefügt


## 4. Alternative Karten-Layer hinzufügen

In [13]:
# Mehrere Karten-Layer
folium.TileLayer('OpenStreetMap').add_to(m)

folium.TileLayer(
    tiles='https://tiles.wmflabs.org/bw-mapnik/{z}/{x}/{y}.png',
    attr='OpenStreetMap',
    name='OSM Schwarzweiß'
).add_to(m)

folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Satellit',
    overlay=False
).add_to(m)

print("✅ Karten-Layer hinzugefügt")

✅ Karten-Layer hinzugefügt


## 5. Interaktive Kontrollen hinzufügen

In [14]:
# Layer-Control
folium.LayerControl(position='topright', collapsed=False).add_to(m)

# Fullscreen
plugins.Fullscreen(position='topright').add_to(m)

print("✅ Controls hinzugefügt")

✅ Controls hinzugefügt


## 6. Legende hinzufügen

In [15]:
# HTML-Legende
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 280px; height: auto; 
     background-color: white; border: 3px solid #2d5016; z-index: 9999; 
     font-size: 13px; padding: 12px; border-radius: 5px; box-shadow: 0 0 15px rgba(0,0,0,0.2);">
     
<h4 style="margin-top: 0; margin-bottom: 10px; color: #2d5016;">🗺️ Legende</h4>

<div style="margin-bottom: 10px;">
  <p style="margin: 5px 0;"><b>Landnutzung:</b></p>
  <p style="margin: 5px 0;">
    <span style="display: inline-block; width: 20px; height: 20px; background: #2d5016; border: 1px solid #1a3009; border-radius: 2px;"></span>
    🌲 Wald
  </p>
  <p style="margin: 5px 0;">
    <span style="display: inline-block; width: 20px; height: 20px; background: #90EE90; border: 1px solid #228B22; border-radius: 2px;"></span>
    🌾 Wiese
  </p>
</div>

<hr style="margin: 8px 0; border: 1px solid #ddd;">

<div style="font-size: 11px;">
  <p style="margin: 5px 0; color: #666;"><b>💡 Tipps:</b></p>
  <ul style="margin: 5px 0; padding-left: 15px;">
    <li>Klick = Info anzeigen</li>
    <li>Layer-Kontrolle oben rechts</li>
    <li>Fullscreen für große Ansicht</li>
  </ul>
</div>

<div style="margin-top: 10px; padding-top: 10px; border-top: 1px solid #ddd; font-size: 10px; color: #888;">
  📊 Baselland Testdaten | {len(gdf)} Lagerplätze
</div>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))
print("✅ Legende hinzugefügt")

✅ Legende hinzugefügt


## 7. Export

In [16]:
output_file = 'lagerplatz_finder_BL_interactive_map.html'

print(f"Exportiere zu: {output_file}...")
m.save(output_file)
print(f"\n✅ Karte gespeichert: {output_file}")

# Dateigröße anzeigen
from pathlib import Path
file_size_mb = Path(output_file).stat().st_size / 1024 / 1024
print(f"📦 Dateigröße: {file_size_mb:.2f} MB")
print(f"\n🌐 Öffne die Datei in deinem Browser!")
print(f"   Oder: jupyter nbconvert --to html --execute --output=map.html notebook.ipynb")

Exportiere zu: lagerplatz_finder_BL_interactive_map.html...

✅ Karte gespeichert: lagerplatz_finder_BL_interactive_map.html
📦 Dateigröße: 11.31 MB

🌐 Öffne die Datei in deinem Browser!
   Oder: jupyter nbconvert --to html --execute --output=map.html notebook.ipynb


## 8. Zusammenfassung

In [17]:
print("""
╔════════════════════════════════════════════════════════════╗
║           📊 STATISTIKEN DER LAGERPLÄTZE                  ║
╚════════════════════════════════════════════════════════════╝
""")

print(f"\n🎯 Allgemein:")
print(f"   Total Lagerplätze: {len(gdf)}")
print(f"   Gesamtfläche: {gdf['flaeche_ha'].sum():,.0f} ha")
print(f"   Durchschnittliche Fläche: {gdf['flaeche_ha'].mean():.1f} ha")

print(f"\n🌍 Landnutzung:")
for landuse, count in gdf['landuse'].value_counts().items():
    pct = count / len(gdf) * 100
    total_area = gdf[gdf['landuse'] == landuse]['flaeche_ha'].sum()
    farbe = FARBEN.get(landuse, {}).get('name', landuse)
    print(f"   {farbe}: {count} ({pct:.1f}%) = {total_area:,.0f} ha")

print(f"\n⭐ Bewertung:")
for bewertung, count in gdf['bewertung'].value_counts().items():
    pct = count / len(gdf) * 100
    print(f"   {bewertung}: {count} ({pct:.1f}%)")

print(f"\n🏆 Filter-Score:")
for score in sorted(gdf['filter_score'].unique(), reverse=True):
    count = len(gdf[gdf['filter_score'] == score])
    pct = count / len(gdf) * 100
    farbe = FILTER_FARBEN.get(score, '#999')
    print(f"   {score}/3 ⭐: {count} ({pct:.1f}%)")

print(f"\n🏗️ Infrastruktur-Nähe:")
print(f"   Nahe Bauernhof: {gdf['nahe_bauernhof'].sum()} ({gdf['nahe_bauernhof'].sum()/len(gdf)*100:.1f}%)")
print(f"   Nahe ÖV: {gdf['nahe_oev'].sum()} ({gdf['nahe_oev'].sum()/len(gdf)*100:.1f}%)")
print(f"   Beides: {((gdf['nahe_bauernhof']) & (gdf['nahe_oev'])).sum()} ({((gdf['nahe_bauernhof']) & (gdf['nahe_oev'])).sum()/len(gdf)*100:.1f}%)")

print(f"\n🎯 Distanzen (Durchschnitt):")
print(f"   Zu Bauernhof: {gdf['dist_bauernhof_m'].mean():.0f} m")
print(f"   Zu ÖV: {gdf['dist_oev_m'].mean():.0f} m")

print("\n" + "="*60)


╔════════════════════════════════════════════════════════════╗
║           📊 STATISTIKEN DER LAGERPLÄTZE                  ║
╚════════════════════════════════════════════════════════════╝


🎯 Allgemein:
   Total Lagerplätze: 1602
   Gesamtfläche: 43,756 ha
   Durchschnittliche Fläche: 27.3 ha

🌍 Landnutzung:
   🌾 Wiese: 1063 (66.4%) = 7,721 ha
   🌲 Wald: 539 (33.6%) = 36,034 ha

⭐ Bewertung:
   geeignet: 1501 (93.7%)
   bedingt geeignet: 98 (6.1%)
   nur Must-have erfüllt: 3 (0.2%)

🏆 Filter-Score:
   2/3 ⭐: 1501 (93.7%)
   1/3 ⭐: 98 (6.1%)
   0/3 ⭐: 3 (0.2%)

🏗️ Infrastruktur-Nähe:
   Nahe Bauernhof: 1559 (97.3%)
   Nahe ÖV: 1541 (96.2%)
   Beides: 1501 (93.7%)

🎯 Distanzen (Durchschnitt):
   Zu Bauernhof: 297 m
   Zu ÖV: 504 m



In [ ]:
print("""
╔════════════════════════════════════════════════════════════╗
║     ✅ NOTEBOOK 3 TEST ERFOLGREICH ABGESCHLOSSEN!         ║
╚════════════════════════════════════════════════════════════╝

📦 Output-Datei:
   → lagerplatz_finder_BL_interactive_map.html

🎨 Karte Features:
   ✓ {count} Lagerplätze visualisiert
   ✓ Nach Landnutzung gefärbt (Wald/Wiese)
   ✓ Pop-ups mit Detailinformationen
   ✓ Infrastruktur-Distanzen angezeigt
   ✓ Multiple Karten-Layer
   ✓ Layer-Kontrolle
   ✓ Fullscreen-Modus
   ✓ Legende mit Erklärungen

🚀 Nächste Schritte:
   1. HTML-Datei im Browser öffnen
   2. Funktionen testen (Zoom, Klick, Layer)
   3. Für ganze Schweiz anpassen (Notebook 2 output)
   4. In Notebook 4 für Pitch verwenden

💡 Pro-Tipps:
   • Zoom auf interessante Region für Demo
   • Verschiedene Karten-Layer zeigen
   • Live-Demo im Pitch = Wow-Effekt!
   • Bei Performance-Problemen: ClusterMarker nutzen

📊 Statistiken siehe oben ⬆️
""".format(count=len(gdf)))


╔════════════════════════════════════════════════════════════╗
║     ✅ NOTEBOOK 3 TEST ERFOLGREICH ABGESCHLOSSEN!         ║
╚════════════════════════════════════════════════════════════╝

📦 Output-Datei:
   → lagerplatz_finder_BL_interactive_map.html

🎨 Karte Features:
   ✓ 1602 Lagerplätze visualisiert
   ✓ Nach Landnutzung gefärbt (Wald/Wiese)
   ✓ Pop-ups mit Detailinformationen
   ✓ Infrastruktur-Distanzen angezeigt
   ✓ Multiple Karten-Layer
   ✓ Layer-Kontrolle
   ✓ Fullscreen-Modus
   ✓ Legende mit Erklärungen

🚀 Nächste Schritte:
   1. HTML-Datei im Browser öffnen
   2. Funktionen testen (Zoom, Klick, Layer)
   3. Für ganze Schweiz anpassen (Notebook 2 output)
   4. In Notebook 4 für Pitch verwenden

💡 Pro-Tipps:
   • Zoom auf interessante Region für Demo
   • Verschiedene Karten-Layer zeigen
   • Live-Demo im Pitch = Wow-Effekt!
   • Bei Performance-Problemen: ClusterMarker nutzen

📊 Statistiken siehe oben ⬆️



: 